# NWRD at the paper's scale: CANet-B4 vs UNet on 128×128 patches

The NWRD paper (Anwar et al., Sensors 2023) trained its UNet on **128×128 patches of photographs
halved in size** with a Lanczos filter (§3.4.1–3.4.2). This notebook trains CANet-B4 and the
paper's UNet architecture at that scale, so accuracy and efficiency are measured where the paper
measured them; the main notebook (`nwrd_experiments.ipynb`) uses 512×512 native-resolution
patches.

**The data.** `abdur548/nwrd-patched` pads each photograph and cuts it, at native resolution,
into 512×512 patches. Each patch is halved (Lanczos) and cut 2×2 into 128×128 tiles, so a tile
covers 256×256 native pixels: the footprint of one of the paper's patches. Tiles go to local disk
(not the saved output) and are rebuilt identically every session, in a few minutes.

**Everything else is the main experiment's protocol**: the same split by photograph, every rust
tile plus 20% of background tiles, 30 epochs, batch 8, AdamW lr 5e-5, checkpoint and threshold
chosen on validation, 3 seeds, photograph-level bootstrap. Only the input scale changes.

**Kaggle setup**
1. *Add Input* → dataset **`abdur548/nwrd-patched`**.
2. *Settings* → Accelerator **GPU T4** (one GPU is used), **Internet on** (pip and ImageNet weights).
3. *Save Version* → **Save & Run All (Commit)**. A session runs for up to 12 h.

**If a session stops at its time budget** (the log ends in `INCOMPLETE`), continue it with
`python scripts/run_on_kaggle.py --experiment 128px --resume --watch` from the repository, or add this
notebook's latest version as an input and commit again. Repeat until the log has no `INCOMPLETE`.

Rough cost on a T4: 2–3 GPU-hours per seed for the two models, so 6–9 GPU-hours in all.

In [ ]:
import glob, os, shutil, subprocess, sys
!pip install -q segmentation-models-pytorch==0.5.0 efficientnet_pytorch==0.7.1 "albumentations>=2.0,<3"
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'enable a GPU accelerator'
os.makedirs('code/nwrd', exist_ok=True)
sys.path.insert(0, os.path.abspath('code'))

In [ ]:
# ── Experiment settings ─────────────────────────────────────────────────────
PRESET = '128px'   # models, input scale and benchmark batches: PRESETS in nwrd/config.py
NUM_WORKERS = 4
# Kaggle kills a session at 12 h. Lower this when the weekly GPU quota has less than ~12 h left,
# so the session stops cleanly (and can be resumed) instead of being killed.
# scripts/run_on_kaggle.py --budget-hours sets it on the pushed copy.
KAGGLE_TIME_BUDGET_HOURS = 11.0
# True = this session must continue earlier runs; stop at once if none are attached rather than
# silently retraining everything. scripts/run_on_kaggle.py --resume sets it on the pushed copy.
EXPECT_RESTORE = False

In [ ]:
# Runs go to /kaggle/working/runs, which Kaggle keeps with each committed version.
OUT = '/kaggle/working/runs' if os.path.isdir('/kaggle/working') else os.path.abspath('runs')
os.makedirs(OUT, exist_ok=True)
print('runs ->', OUT, '| time budget', KAGGLE_TIME_BUDGET_HOURS, 'h')

### Pipeline source (generated from `nwrd/` by `scripts/build_notebooks.py`; edit there)

In [ ]:
%%writefile code/nwrd/__init__.py
"""Wheat stripe-rust segmentation on the NWRD dataset: CANet-B4 against the paper's baselines."""

__version__ = '1.0.0'


In [ ]:
%%writefile code/nwrd/config.py
"""Experiment protocol. Every model in an experiment is trained with exactly these settings.

Each field is also a command-line flag of `python -m nwrd.run` (`data_root` -> `--data-root`).
A preset sets the fields that define one experiment; flags given explicitly override it.
"""

from dataclasses import asdict, dataclass, field


@dataclass
class Config:
    data_root: str = ''
    out_dir: str = 'runs'
    models: list[str] = field(
        default_factory=lambda: ['canet_b4', 'unet', 'deeplabv3plus_r50', 'unet_tuned']
    )
    seeds: list[int] = field(default_factory=lambda: [42, 43, 44])
    data_seed: int = 42  # background subsample; fixed so all runs see the same patches

    img_size: int = 512
    # >0: train and evaluate on tiles re-cut from every patch the way the NWRD paper cut its
    # patches: resized by tile_scale (Lanczos), then split into tile_size squares. 128 and 0.5
    # reproduce the paper's patches (nwrd/tiles.py). img_size must then equal tile_size.
    tile_size: int = 0
    tile_scale: float = 0.5
    tile_dir: str = ''  # where tiles are written; default: under the temp directory
    bg_keep_ratio: float = 0.20
    epochs: int = 30
    freeze_encoder_epochs: int = 3
    batch_size: int = 8
    num_workers: int = 2
    lr: float = 5e-5
    # 'unet_tuned' only: the shared lr suits fine-tuning pretrained encoders and may
    # under-train a from-scratch UNet, so this control variant gets its own.
    unet_tuned_lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    dice_weight: float = 0.7
    bce_weight: float = 0.3
    dropout: float = 0.5  # CANet head dropout; the other models have none
    sched_t0: int = 5
    sched_tmult: int = 2
    sched_eta_min: float = 1e-7
    amp: bool = True
    pretrained: bool = True

    # threshold chosen on val over this grid, then frozen and applied to test
    thr_lo: float = 0.05
    thr_hi: float = 0.95
    thr_step: float = 0.01

    # efficiency benchmark: models to measure (empty = the trained models), fp16 batch sizes
    bench_models: list[str] = field(default_factory=list)
    bench_batches: list[int] = field(default_factory=lambda: [8])
    bench_quick: bool = False  # few timing iterations (smoke tests)

    time_budget_hours: float = 0.0  # >0: stop cleanly before a platform session limit
    limit: int = 0  # >0: use only N patches per split (smoke tests)

    def validate(self) -> None:
        if self.tile_size and self.img_size != self.tile_size:
            raise ValueError(
                f'img_size {self.img_size} would resize the {self.tile_size}px tiles; '
                f'set --img-size {self.tile_size}'
            )

    def to_dict(self) -> dict:
        return asdict(self)


# The two experiments. `main` is the protocol as defined above; `128px` changes only the input
# scale, to the NWRD paper's 128px patches of half-size photographs, so 512px vs 128px is a
# one-variable comparison. fp16 throughput at 128px is also measured at the paper's batch of 64.
PRESETS = {
    'main': {},
    '128px': {
        'models': ['canet_b4', 'unet', 'deeplabv3plus_r50'],
        'img_size': 128,
        'tile_size': 128,
        'tile_scale': 0.5,
        'bench_batches': [8, 64],
    },
}


In [ ]:
%%writefile code/nwrd/data.py
"""NWRD patch dataset: split discovery, background subsampling, transforms.

The dataset (Kaggle `abdur548/nwrd-patched`) holds 512x512 native-resolution patches of the 100
NWRD photographs, split by photograph: <root>/{train,val,test}/{images,masks}/<photo>_patch_*.png.
"""

import glob
import json
import os
import random

import cv2
import numpy as np
import torch

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet statistics
DATASET_DIRNAME = 'wheat_rust_patches'


def find_data_root(hint: str | None = None) -> str:
    """Returns the directory containing train/ val/ test/, searching `hint`, then Kaggle's
    input mount."""
    candidates = [hint] if hint else []
    candidates.append('/kaggle/input')
    for base in candidates:
        if os.path.isdir(os.path.join(base, 'train', 'images')):
            return base
        hits = glob.glob(os.path.join(base, '**', DATASET_DIRNAME), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(
        f'no {DATASET_DIRNAME}/ found under {candidates}; attach the Kaggle dataset '
        f'abdur548/nwrd-patched or pass --data-root'
    )


def split_dirs(root: str, split: str) -> tuple[str, str]:
    return os.path.join(root, split, 'images'), os.path.join(root, split, 'masks')


def read_mask(path: str) -> np.ndarray:
    return (cv2.imread(path, cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)


def mask_stats(root: str, split: str, cache_dir: str | None = None) -> dict:
    """Per-file rust pixel counts for a split, cached as JSON (reading 8k masks is slow)."""
    cache = os.path.join(cache_dir, f'mask_stats_{split}.json') if cache_dir else None
    if cache and os.path.exists(cache):
        with open(cache) as f:
            return json.load(f)
    _, mask_dir = split_dirs(root, split)
    stats = {}
    for name in sorted(os.listdir(mask_dir)):
        m = read_mask(os.path.join(mask_dir, name))
        stats[name] = {'pos': int(m.sum()), 'total': int(m.size)}
    if cache:
        os.makedirs(cache_dir, exist_ok=True)
        with open(cache, 'w') as f:
            json.dump(stats, f)
    return stats


def select_train_files(stats: dict, bg_keep_ratio: float, seed: int) -> tuple[list[str], dict]:
    """All rust-positive patches plus a seeded random fraction of background-only ones.

    The seed is the *data* seed, fixed across models so every model sees the same patches."""
    disease = sorted(f for f, s in stats.items() if s['pos'] > 0)
    background = sorted(f for f, s in stats.items() if s['pos'] == 0)
    kept = random.Random(seed).sample(background, int(len(background) * bg_keep_ratio))
    info = {'disease': len(disease), 'bg_kept': len(kept), 'bg_total': len(background)}
    return disease + sorted(kept), info


def pos_weight_from_stats(stats: dict, files: list[str]) -> float:
    """Background-to-rust pixel ratio over `files`: the BCE positive-class weight."""
    pos = sum(stats[f]['pos'] for f in files)
    total = sum(stats[f]['total'] for f in files)
    return (total - pos) / max(pos, 1)


def train_transform(size: int):
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    return A.Compose(
        [
            A.Resize(size, size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.ElasticTransform(p=0.5),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ]
    )


def eval_transform(size: int):
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    return A.Compose([A.Resize(size, size), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])


class RustDataset(torch.utils.data.Dataset):
    """(image tensor, mask tensor of shape (1, H, W), index) for each file of a split."""

    def __init__(self, root: str, split: str, files, transform):
        self.img_dir, self.mask_dir = split_dirs(root, split)
        self.files, self.transform = list(files), transform

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        name = self.files[idx]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, name)), cv2.COLOR_BGR2RGB)
        out = self.transform(image=img, mask=read_mask(os.path.join(self.mask_dir, name)))
        return out['image'], out['mask'].unsqueeze(0), idx


In [ ]:
%%writefile code/nwrd/tiles.py
"""Re-cuts the 512px NWRD patches at the NWRD paper's scale.

The paper (Anwar et al., Sensors 2023, §3.4.1-3.4.2) halves every photograph with a Lanczos
filter, then cuts 128x128 patches. The Kaggle dataset pads each photograph and cuts it, at
native resolution, into 512x512 patches. Halving a patch and cutting it 2x2 therefore gives
128px tiles covering the same 256x256 native pixels as one of the paper's patches.

Each tile keeps its patch's name plus a _t<row><col> suffix, so compare.source_photo still
groups tiles by photograph. The output is deterministic, and a finished build is reused.

    python -m nwrd.tiles --src /path/to/wheat_rust_patches --dst /tmp/nwrd_tiles
"""

import argparse
import json
import os
from concurrent.futures import ProcessPoolExecutor

import cv2
import numpy as np

from .data import split_dirs

SPLITS = ('train', 'val', 'test')


def tile_name(name: str, row: int, col: int) -> str:
    stem, ext = os.path.splitext(name)
    return f'{stem}_t{row}{col}{ext}'


def downsample_and_tile(img: np.ndarray, mask: np.ndarray, scale: float, tile: int):
    """Yields (row, col, image tile, mask tile). The image is resized with Lanczos, as in the
    paper. A mask pixel is rust when at least half of the area it replaces was rust."""
    h, w = mask.shape
    size = (round(w * scale), round(h * scale))
    if size[0] % tile or size[1] % tile:
        raise ValueError(
            f'{w}x{h} scaled by {scale} is {size[0]}x{size[1]}, '
            f'not a whole number of {tile}px tiles'
        )
    img = cv2.resize(img, size, interpolation=cv2.INTER_LANCZOS4)
    rust = cv2.resize((mask > 127).astype(np.float32), size, interpolation=cv2.INTER_AREA)
    mask = (rust >= 0.5).astype(np.uint8) * 255
    for r in range(size[1] // tile):
        for c in range(size[0] // tile):
            ys, xs = slice(r * tile, (r + 1) * tile), slice(c * tile, (c + 1) * tile)
            yield r, c, img[ys, xs], mask[ys, xs]


def _tile_patch(job) -> tuple[int, int, int, int, int]:
    """Tiles one patch; returns (tiles, rust tiles, rust px before, px before, rust px after)."""
    name, src_img, src_mask, dst_img, dst_mask, scale, tile = job
    img = cv2.imread(os.path.join(src_img, name), cv2.IMREAD_COLOR)
    mask = cv2.imread(os.path.join(src_mask, name), cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        raise FileNotFoundError(f'cannot read {name} from {src_img} and {src_mask}')
    n = rust_tiles = rust_px = 0
    for r, c, ti, tm in downsample_and_tile(img, mask, scale, tile):
        out = tile_name(name, r, c)
        cv2.imwrite(os.path.join(dst_img, out), ti)
        cv2.imwrite(os.path.join(dst_mask, out), tm)
        pos = int((tm > 0).sum())
        n, rust_tiles, rust_px = n + 1, rust_tiles + (pos > 0), rust_px + pos
    return n, rust_tiles, int((mask > 127).sum()), int(mask.size), rust_px


def build_tiles(
    src: str, dst: str, scale: float = 0.5, tile: int = 128, workers: int | None = None, log=print
) -> str:
    """Writes <dst>/<split>/{images,masks}/ from the patches under `src` and returns `dst`.

    A finished build (marked by <dst>/tiling.json) with the same settings is reused, since
    tiling is deterministic. A build with other settings is refused rather than overwritten."""
    if os.path.abspath(dst) == os.path.abspath(src):
        raise ValueError('the tile directory must differ from the patch directory')
    params = {'source': os.path.abspath(src), 'scale': scale, 'tile': tile}
    marker = os.path.join(dst, 'tiling.json')
    if os.path.exists(marker):
        with open(marker) as f:
            done = json.load(f)
        if {k: done.get(k) for k in params} != params:
            raise RuntimeError(f'{dst} holds tiles built with {done}; use another --tile-dir')
        log(f'tiles: reusing {dst}')
        return dst

    stats = {}
    for split in SPLITS:
        src_img, src_mask = split_dirs(src, split)
        dst_img, dst_mask = split_dirs(dst, split)
        os.makedirs(dst_img, exist_ok=True)
        os.makedirs(dst_mask, exist_ok=True)
        jobs = [
            (f, src_img, src_mask, dst_img, dst_mask, scale, tile)
            for f in sorted(os.listdir(src_img))
        ]
        if workers == 1:
            res = [_tile_patch(j) for j in jobs]
        else:
            with ProcessPoolExecutor(workers) as ex:
                res = list(ex.map(_tile_patch, jobs, chunksize=32))
        n, rust_tiles, pos_before, px_before, pos_after = np.sum(res, axis=0).tolist()
        stats[split] = {
            'patches': len(jobs),
            'tiles': n,
            'rust_tiles': rust_tiles,
            'rust_pixel_fraction_patches': pos_before / px_before,
            'rust_pixel_fraction_tiles': pos_after / (n * tile * tile),
        }
        log(f'tiles: {split} {len(jobs)} patches -> {n} tiles ({rust_tiles} with rust)')
    with open(marker, 'w') as f:
        json.dump({**params, 'splits': stats}, f, indent=1)
    return dst


def main(argv=None) -> None:
    p = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter
    )
    p.add_argument('--src', required=True, help='directory holding train/ val/ test/')
    p.add_argument('--dst', required=True)
    p.add_argument('--scale', type=float, default=0.5)
    p.add_argument('--tile', type=int, default=128)
    p.add_argument('--workers', type=int, default=None)
    a = p.parse_args(argv)
    build_tiles(a.src, a.dst, a.scale, a.tile, a.workers)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile code/nwrd/models.py
"""Model definitions: CANet-B4 (ours), the NWRD paper's UNet, and the DeepLabV3+ baselines.

Module names inside CANet match the checkpoints trained since the sprint-2 notebook, so every
existing checkpoint loads unchanged.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import Config

MODELS = {
    'canet_b4': 'CANet, EfficientNet-B4 encoder (ours)',
    'deeplabv3plus_r50': 'DeepLabV3+, ResNet-50 encoder (baseline)',
    'deeplabv3plus_b4': 'DeepLabV3+, EfficientNet-B4 encoder (same-encoder ablation)',
    'unet': 'UNet, 4 encoder / 4 decoder blocks, from scratch (NWRD paper baseline)',
    'unet_tuned': 'UNet (paper architecture) with a from-scratch lr (config.unet_tuned_lr)',
}


class ConvBNReLU(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, p: int = 1, d: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, padding=p, dilation=d, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ContextFlow(nn.Module):
    """One CAM information flow: a shallow encoder-decoder at a given downsample scale."""

    def __init__(self, in_ch: int, out_ch: int, scale: int):
        super().__init__()
        self.scale = scale
        self.encode = ConvBNReLU(in_ch, out_ch)
        self.decode = ConvBNReLU(out_ch, out_ch)

    def forward(self, x):
        h, w = x.shape[-2:]
        # A window larger than the map would pool it to nothing: a 128px input gives a 4x4 map at
        # 1/32, which scale 8 cannot pool. Capped, that flow pools globally. Maps of 8x8 and up
        # (inputs of 256px and up) are untouched.
        k = min(self.scale, h, w)
        xd = F.avg_pool2d(x, k) if k > 1 else x
        f = self.decode(self.encode(xd))
        return F.interpolate(f, (h, w), mode='bilinear', align_corners=False) if k > 1 else f


class AttentionFusion(nn.Module):
    """Re-fusion: a per-pixel, per-channel sigmoid gate computed from the fused features."""

    def __init__(self, in_ch: int):
        super().__init__()
        self.att = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch // 4, in_ch, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.att(x)


class CAM(nn.Module):
    """Context aggregation module: a serial dilated global flow plus parallel context
    flows at scales 2/4/8, pre-fused by concatenation and re-fused by the attention gate."""

    def __init__(self, in_ch: int, out_ch: int = 256):
        super().__init__()
        self.global_flow = nn.Sequential(
            ConvBNReLU(in_ch, out_ch, k=3, p=2, d=2),
            ConvBNReLU(out_ch, out_ch, k=3, p=4, d=4),
            ConvBNReLU(out_ch, out_ch, k=3, p=8, d=8),
        )
        self.context_flows = nn.ModuleList(
            [
                ContextFlow(in_ch, out_ch, 2),
                ContextFlow(in_ch, out_ch, 4),
                ContextFlow(in_ch, out_ch, 8),
            ]
        )
        self.pre_fusion = ConvBNReLU(out_ch * 4, out_ch, k=1, p=0)
        self.re_fusion = AttentionFusion(out_ch)
        self.out_conv = ConvBNReLU(out_ch, out_ch)

    def forward(self, x):
        gf = self.global_flow(x)
        cfs = [cf(x) for cf in self.context_flows]
        fused = self.pre_fusion(torch.cat([gf] + cfs, dim=1))
        return self.out_conv(self.re_fusion(fused))


class AsymmetricDecoder(nn.Module):
    """Fuses upsampled 1/32 context with reduced 1/4 low-level features."""

    def __init__(self, high_ch: int, low_ch: int, out_ch: int = 128):
        super().__init__()
        self.low_reduce = ConvBNReLU(low_ch, 48, k=1, p=0)
        self.fuse = nn.Sequential(ConvBNReLU(high_ch + 48, out_ch), ConvBNReLU(out_ch, out_ch))

    def forward(self, high, low):
        high_up = F.interpolate(high, low.shape[-2:], mode='bilinear', align_corners=False)
        return self.fuse(torch.cat([high_up, self.low_reduce(low)], dim=1))


class CANet(nn.Module):
    """CANet with an EfficientNet-B4 encoder: CAM on the 1/32 features, fused with the 1/4
    features by an asymmetric decoder, then a 1/4-resolution head upsampled x4."""

    has_pretrained_encoder = True  # its encoder is frozen for the first epochs (train.py)

    def __init__(self, dropout: float = Config.dropout, pretrained: bool = True):
        super().__init__()
        from efficientnet_pytorch import EfficientNet

        self.encoder = (
            EfficientNet.from_pretrained('efficientnet-b4')
            if pretrained
            else EfficientNet.from_name('efficientnet-b4')
        )
        self.reduce = ConvBNReLU(1792, 512, k=1, p=0)
        self.cam = CAM(512, 256)
        self.decoder = AsymmetricDecoder(256, 32, 128)  # reduction_2: 32ch at 1/4
        self.dropout = nn.Dropout2d(dropout)
        self.head = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        # One encoder pass yields both the 1/32 features (reduction_6, which equals
        # extract_features(x)) and the 1/4 features (reduction_2).
        ep = self.encoder.extract_endpoints(x)
        enc, low = ep['reduction_6'], ep['reduction_2']
        x_cam = self.cam(self.reduce(enc))
        x_dec = self.dropout(self.decoder(x_cam, low))
        return F.interpolate(self.head(x_dec), scale_factor=4, mode='bilinear', align_corners=False)


class _DoubleConv(nn.Sequential):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class UNet(nn.Module):
    """Plain UNet as used by the NWRD paper (Anwar et al., Sensors 2023): 4 encoder and 4
    decoder blocks, 64-1024 channels, transposed-conv upsampling, no pretraining."""

    has_pretrained_encoder = False  # trained from scratch, so never frozen

    def __init__(self, base: int = 64):
        super().__init__()
        ch = [base * 2**i for i in range(5)]
        self.inc = _DoubleConv(3, ch[0])
        self.encoder = nn.ModuleList(_DoubleConv(ch[i], ch[i + 1]) for i in range(4))
        self.up = nn.ModuleList(
            nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(4))
        )
        self.decoder = nn.ModuleList(_DoubleConv(ch[i] * 2, ch[i]) for i in reversed(range(4)))
        self.head = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = [self.inc(x)]
        for down in self.encoder:
            skips.append(down(F.max_pool2d(skips[-1], 2)))
        x = skips.pop()
        for up, dec in zip(self.up, self.decoder, strict=True):
            x = dec(torch.cat([skips.pop(), up(x)], 1))
        return self.head(x)


def build_model(name: str, pretrained: bool = True, dropout: float = Config.dropout) -> nn.Module:
    """All models return raw logits of shape (B, 1, H, W) and expose `.encoder`.

    A model whose class sets `has_pretrained_encoder = False` is never frozen; the smp
    baselines have ImageNet encoders and default to being frozen like CANet's."""
    if name == 'canet_b4':
        return CANet(dropout=dropout, pretrained=pretrained)
    if name in ('unet', 'unet_tuned'):
        return UNet()
    if name in ('deeplabv3plus_r50', 'deeplabv3plus_b4'):
        import segmentation_models_pytorch as smp

        encoder = 'resnet50' if name.endswith('r50') else 'efficientnet-b4'
        return smp.DeepLabV3Plus(
            encoder_name=encoder, encoder_weights='imagenet' if pretrained else None, classes=1
        )
    raise ValueError(f'unknown model {name!r}; choose from {sorted(MODELS)}')


def load_checkpoint(model: nn.Module, path: str, map_location='cpu') -> dict:
    """Loads a checkpoint written by the pipeline ({'model': state_dict, ...}) into `model`
    and returns the whole checkpoint. Only tensors and plain data are unpickled."""
    ckpt = torch.load(path, map_location=map_location, weights_only=True)
    model.load_state_dict(ckpt['model'])
    return ckpt


In [ ]:
%%writefile code/nwrd/metrics.py
"""Pixel metrics pooled over a whole split.

The sprint notebooks averaged IoU per batch, which scores a correctly predicted all-background
batch as 0 and depresses the curve (~0.58 vs ~0.81 pooled). Everything here pools the confusion
counts over every pixel of the split, then computes the ratios once.

Confusion counts are always ordered (tp, fp, fn, tn).
"""

import numpy as np
import torch

EPS = 1e-12


def scores(tp, fp, fn, tn) -> dict[str, float]:
    tp, fp, fn, tn = (float(v) for v in (tp, fp, fn, tn))
    return {
        'IoU': tp / (tp + fp + fn + EPS),
        'Dice': 2 * tp / (2 * tp + fp + fn + EPS),
        'Precision': tp / (tp + fp + EPS),
        'Recall': tp / (tp + fn + EPS),
        'Specificity': tn / (tn + fp + EPS),
        'Accuracy': (tp + tn) / (tp + tn + fp + fn + EPS),
    }


def iou(counts: np.ndarray) -> np.ndarray:
    """IoU of every row of an array of (..., 4) confusion counts."""
    tp, fp, fn = counts[..., 0], counts[..., 1], counts[..., 2]
    return tp / np.maximum(tp + fp + fn, 1)


def dice(counts: np.ndarray) -> np.ndarray:
    """Dice of every row of an array of (..., 4) confusion counts."""
    tp, fp, fn = counts[..., 0], counts[..., 1], counts[..., 2]
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def per_image_counts(probs: torch.Tensor, target: torch.Tensor, threshold: float) -> np.ndarray:
    """(B, 4) int64 array of tp, fp, fn, tn per image. Decision rule: prob >= threshold."""
    pred = probs >= threshold
    tgt = target > 0.5
    dims = tuple(range(1, probs.dim()))
    tp = (pred & tgt).sum(dims)
    fp = (pred & ~tgt).sum(dims)
    fn = (~pred & tgt).sum(dims)
    tn = (~pred & ~tgt).sum(dims)
    return torch.stack([tp, fp, fn, tn], 1).cpu().numpy().astype(np.int64)


class ProbHistogram:
    """Histograms of predicted probability for rust and background pixels.

    Gives exact pooled confusion counts at any threshold on a 1/bins grid without keeping
    every prediction in memory."""

    def __init__(self, bins: int = 1000):
        self.bins = bins
        self.pos = np.zeros(bins, np.int64)
        self.neg = np.zeros(bins, np.int64)

    def update(self, probs: torch.Tensor, target: torch.Tensor) -> None:
        q = (probs.float() * self.bins).long().clamp_(0, self.bins - 1).flatten()
        t = (target > 0.5).flatten()
        self.pos += torch.bincount(q[t], minlength=self.bins).cpu().numpy()
        self.neg += torch.bincount(q[~t], minlength=self.bins).cpu().numpy()

    def counts(self, threshold: float) -> tuple[int, int, int, int]:
        k = int(round(threshold * self.bins))
        tp, fp = self.pos[k:].sum(), self.neg[k:].sum()
        return tp, fp, self.pos.sum() - tp, self.neg.sum() - fp

    def sweep(self, lo: float = 0.05, hi: float = 0.95, step: float = 0.01) -> list[dict]:
        rows = []
        for t in np.round(np.arange(lo, hi + step / 2, step), 4):
            s = scores(*self.counts(t))
            rows.append({'threshold': float(t), **s})
        return rows

    def best_threshold(self, **kw) -> tuple[float, list[dict]]:
        """IoU = Dice / (2 - Dice), so maximising either picks the same threshold."""
        rows = self.sweep(**kw)
        return max(rows, key=lambda r: r['IoU'])['threshold'], rows


In [ ]:
%%writefile code/nwrd/train.py
"""Training for one (model, seed) run. Resumable from last.pt."""

import json
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn

from . import data as D
from .config import Config
from .metrics import ProbHistogram, scores
from .models import build_model


class CombinedLoss(nn.Module):
    """dice_w * Dice loss + bce_w * BCE, both on logits; BCE weights rust pixels by pos_weight."""

    def __init__(self, dice_w: float, bce_w: float, pos_weight: torch.Tensor):
        super().__init__()
        import segmentation_models_pytorch as smp

        self.dice_w, self.bce_w = dice_w, bce_w
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, target):
        return self.dice_w * self.dice(logits, target) + self.bce_w * self.bce(logits, target)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _limit(files: list[str], n: int) -> list[str]:
    return files[:n] if n else files


def _seed_worker(worker_id: int) -> None:
    """Seeds a DataLoader worker's augmentation from the seed PyTorch derives for that worker
    from the loader's generator, so augmentation depends on (seed, epoch, worker) only.
    Albumentations 2 seeds its transforms from the OS otherwise, not from the global seed."""
    info = torch.utils.data.get_worker_info()
    info.dataset.transform.set_random_seed(info.seed % 2**32)


def make_loaders(cfg: Config, cache_dir: str):
    """Returns the train and val loaders, data info, and the train generator (reseed it every
    epoch)."""
    root = cfg.data_root
    train_stats = D.mask_stats(root, 'train', cache_dir)
    train_files, sel = D.select_train_files(train_stats, cfg.bg_keep_ratio, cfg.data_seed)
    train_files = _limit(train_files, cfg.limit)
    val_files = _limit(sorted(D.mask_stats(root, 'val', cache_dir)), cfg.limit)
    pos_weight = D.pos_weight_from_stats(train_stats, train_files)

    kw = dict(
        batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available()
    )
    g = torch.Generator()
    train_loader = torch.utils.data.DataLoader(
        D.RustDataset(root, 'train', train_files, D.train_transform(cfg.img_size)),
        shuffle=True,
        drop_last=True,
        generator=g,
        worker_init_fn=_seed_worker,
        **kw,
    )
    val_loader = torch.utils.data.DataLoader(
        D.RustDataset(root, 'val', val_files, D.eval_transform(cfg.img_size)), shuffle=False, **kw
    )
    info = {**sel, 'train_used': len(train_files), 'val': len(val_files), 'pos_weight': pos_weight}
    return train_loader, val_loader, info, g


def run_validation(model, loader, criterion, device, amp: bool):
    """Returns the mean validation loss and the probability histogram of the split."""
    model.eval()
    hist, loss_sum, n = ProbHistogram(), 0.0, 0
    with torch.no_grad():
        for imgs, masks, _ in loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            with torch.autocast(device.type, enabled=amp):
                logits = model(imgs)
            loss_sum += criterion(logits.float(), masks).item() * imgs.size(0)
            n += imgs.size(0)
            hist.update(torch.sigmoid(logits.float()), masks)
    return loss_sum / n, hist


def train_run(
    cfg: Config, model_name: str, seed: int, run_dir: str, log=print, session_start=None
) -> bool:
    """Trains one run to cfg.epochs, keeping the checkpoint with the best pooled val IoU at
    threshold 0.5 as best.pt. Returns False if the time budget stopped it (resume later)."""
    os.makedirs(run_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = cfg.amp and device.type == 'cuda'
    torch.backends.cudnn.benchmark = True
    set_seed(seed)

    train_loader, val_loader, info, gen = make_loaders(cfg, os.path.join(cfg.out_dir, '_cache'))
    log(f'[{model_name} s{seed}] data: {info}')
    model = build_model(model_name, pretrained=cfg.pretrained, dropout=cfg.dropout).to(device)
    criterion = CombinedLoss(
        cfg.dice_weight, cfg.bce_weight, torch.tensor([info['pos_weight']], device=device)
    )
    lr = cfg.unet_tuned_lr if model_name == 'unet_tuned' else cfg.lr
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=cfg.sched_t0, T_mult=cfg.sched_tmult, eta_min=cfg.sched_eta_min
    )
    scaler = torch.amp.GradScaler(device.type, enabled=amp)

    history = {
        k: []
        for k in (
            'train_loss',
            'val_loss',
            'val_iou',
            'val_dice',
            'val_iou_best_thr',
            'lr',
            'epoch_seconds',
        )
    }
    start_epoch, best_iou = 0, -1.0
    last_path, best_path = os.path.join(run_dir, 'last.pt'), os.path.join(run_dir, 'best.pt')
    if os.path.exists(last_path):
        # Load on the CPU: torch.set_rng_state only accepts a CPU tensor, and load_state_dict
        # moves model and optimizer state onto the model's device by itself.
        ck = torch.load(last_path, map_location='cpu', weights_only=True)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        scaler.load_state_dict(ck['scaler'])
        torch.set_rng_state(ck['rng_cpu'])
        if device.type == 'cuda' and ck.get('rng_cuda') is not None:
            torch.cuda.set_rng_state_all(ck['rng_cuda'])
        start_epoch, best_iou, history = ck['epoch'], ck['best_iou'], ck['history']
        log(f'[{model_name} s{seed}] resumed at epoch {start_epoch}')

    t_start = time.time()
    first_in_session = session_start is None
    session_start = t_start if first_in_session else session_start
    for epoch in range(start_epoch, cfg.epochs):
        # The budget covers the whole session, not this run. The session's first run always
        # gets one epoch, so every session makes progress; the headroom between the budget
        # and the platform limit covers that epoch when no per-epoch estimate exists yet.
        if cfg.time_budget_hours and (epoch > start_epoch or not first_in_session):
            done = epoch - start_epoch
            per_epoch = (time.time() - t_start) / done if done else 0.0
            if time.time() - session_start + 1.5 * per_epoch > cfg.time_budget_hours * 3600:
                log(f'[{model_name} s{seed}] time budget reached at epoch {epoch}; resume later')
                return False

        # Freezing only makes sense for an ImageNet-pretrained encoder.
        frozen = epoch < cfg.freeze_encoder_epochs and getattr(
            model, 'has_pretrained_encoder', True
        )
        for p in model.encoder.parameters():
            p.requires_grad = not frozen

        # Shuffle order and augmentation depend on (seed, epoch) only, so a run is reproducible
        # from its seed and a resumed run sees the batches it would have seen uninterrupted.
        # Worker processes reseed their augmentation from `gen` (_seed_worker); without
        # workers, the augmentation runs here.
        gen.manual_seed(seed * 10_000 + epoch)
        train_loader.dataset.transform.set_random_seed(seed * 10_000 + epoch)
        t0 = time.time()
        model.train()
        loss_sum, n = 0.0, 0
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=amp):
                logits = model(imgs)
            loss = criterion(logits.float(), masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * imgs.size(0)
            n += imgs.size(0)
        lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        val_loss, hist = run_validation(model, val_loader, criterion, device, amp)
        val = scores(*hist.counts(0.5))
        _, sweep = hist.best_threshold(lo=cfg.thr_lo, hi=cfg.thr_hi, step=cfg.thr_step)
        for k, v in (
            ('train_loss', loss_sum / n),
            ('val_loss', val_loss),
            ('val_iou', val['IoU']),
            ('val_dice', val['Dice']),
            ('val_iou_best_thr', max(r['IoU'] for r in sweep)),
            ('lr', lr),
            ('epoch_seconds', time.time() - t0),
        ):
            history[k].append(v)
        log(
            f'[{model_name} s{seed}] ep {epoch + 1:02d}/{cfg.epochs} '
            f'{"(enc frozen) " if frozen else ""}train {loss_sum / n:.4f} val {val_loss:.4f} '
            f'IoU@0.5 {val["IoU"]:.4f} Dice@0.5 {val["Dice"]:.4f} '
            f'({history["epoch_seconds"][-1]:.0f}s)'
        )

        # Model selection: pooled val IoU at the fixed 0.5 threshold.
        if val['IoU'] > best_iou:
            best_iou = val['IoU']
            torch.save(
                {
                    'model': model.state_dict(),
                    'epoch': epoch + 1,
                    'val_iou@0.5': best_iou,
                    'model_name': model_name,
                    'seed': seed,
                },
                best_path,
            )
        torch.save(
            {
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict(),
                'rng_cpu': torch.get_rng_state(),
                'rng_cuda': torch.cuda.get_rng_state_all() if device.type == 'cuda' else None,
                'epoch': epoch + 1,
                'best_iou': best_iou,
                'history': history,
            },
            last_path,
        )
        with open(os.path.join(run_dir, 'history.json'), 'w') as f:
            json.dump({'history': history, 'data': info, 'best_val_iou@0.5': best_iou}, f, indent=1)

    os.remove(last_path)  # optimizer state is ~2/3 of the file; best.pt is what we keep
    with open(os.path.join(run_dir, 'TRAINED'), 'w') as f:
        f.write(f'best val IoU@0.5 {best_iou:.6f}\n')
    return True


In [ ]:
%%writefile code/nwrd/evaluate.py
"""Evaluation of a trained run.

Protocol: the binarisation threshold is chosen on the validation split, frozen, and applied
once to the held-out test split. Test metrics are also reported at the fixed 0.5 threshold,
which needs no tuning at all.
"""

import json
import os

import numpy as np
import torch

from . import data as D
from .config import Config
from .metrics import ProbHistogram, per_image_counts, scores
from .models import build_model, load_checkpoint


def predict_split(model, cfg: Config, split: str, device, thresholds=()):
    """Returns the probability histogram, per-image counts at each threshold, and files."""
    files = sorted(os.listdir(D.split_dirs(cfg.data_root, split)[0]))
    files = files[: cfg.limit] if cfg.limit else files
    loader = torch.utils.data.DataLoader(
        D.RustDataset(cfg.data_root, split, files, D.eval_transform(cfg.img_size)),
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        shuffle=False,
    )
    amp = cfg.amp and device.type == 'cuda'
    hist = ProbHistogram()
    counts = {t: [] for t in thresholds}
    model.eval()
    with torch.no_grad():
        for imgs, masks, _ in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            with torch.autocast(device.type, enabled=amp):
                probs = torch.sigmoid(model(imgs).float())
            hist.update(probs, masks)
            for t in thresholds:
                counts[t].append(per_image_counts(probs, masks, t))
    return hist, {t: np.concatenate(v) for t, v in counts.items()}, files


def evaluate_run(cfg: Config, model_name: str, run_dir: str, log=print) -> dict:
    """Picks the threshold on val, scores test, and writes metrics.json,
    val_threshold_sweep.json and test_counts.npz into `run_dir`."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_model(model_name, pretrained=False, dropout=cfg.dropout).to(device)
    ckpt = load_checkpoint(model, os.path.join(run_dir, 'best.pt'), map_location=device)

    val_hist, _, _ = predict_split(model, cfg, 'val', device)
    thr, val_sweep = val_hist.best_threshold(lo=cfg.thr_lo, hi=cfg.thr_hi, step=cfg.thr_step)
    at_edge = thr in (cfg.thr_lo, cfg.thr_hi)

    test_hist, test_counts, test_files = predict_split(
        model, cfg, 'test', device, thresholds=tuple(dict.fromkeys((thr, 0.5)))
    )
    rust = np.array([c[0] + c[2] > 0 for c in test_counts[thr]])
    res = {
        'model': model_name,
        'best_epoch': ckpt.get('epoch'),
        'threshold': thr,
        'threshold_at_grid_edge': bool(at_edge),
        'val@thr': scores(*val_hist.counts(thr)),
        'val@0.5': scores(*val_hist.counts(0.5)),
        'test@thr': scores(*test_hist.counts(thr)),
        'test@0.5': scores(*test_hist.counts(0.5)),
        # NWRD paper Table 4 protocol: background-only patches removed from the test set.
        'test_rust_patches@thr': scores(*test_counts[thr][rust].sum(0)),
        'test_images': len(test_files),
        'test_rust_images': int(rust.sum()),
        'test_rust_pixel_fraction': float(
            test_hist.pos.sum() / (test_hist.pos.sum() + test_hist.neg.sum())
        ),
    }
    with open(os.path.join(run_dir, 'metrics.json'), 'w') as f:
        json.dump(res, f, indent=2)
    with open(os.path.join(run_dir, 'val_threshold_sweep.json'), 'w') as f:
        json.dump(val_sweep, f)
    # Per-image confusion counts drive the paired bootstrap in compare.py.
    np.savez_compressed(
        os.path.join(run_dir, 'test_counts.npz'),
        files=np.array(test_files),
        at_thr=test_counts[thr],
        at_05=test_counts[0.5],
    )
    t = res['test@thr']
    log(
        f'[{model_name} {os.path.basename(run_dir)}] thr {thr:.2f}'
        f'{" (GRID EDGE)" if at_edge else ""} | test IoU {t["IoU"]:.4f} Dice {t["Dice"]:.4f} '
        f'P {t["Precision"]:.4f} R {t["Recall"]:.4f}'
    )
    return res


def plot_qualitative(cfg: Config, run_dirs: dict[str, str], path: str, n: int = 6, seed: int = 0):
    """Test patches with rust: image, ground truth, then each model's prediction at its own
    val-selected threshold. `run_dirs` maps model name -> run directory."""
    import matplotlib

    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    counts = np.load(os.path.join(next(iter(run_dirs.values())), 'test_counts.npz'))
    rust = [i for i, c in enumerate(counts['at_thr']) if c[0] + c[2] > 0]
    idx = sorted(np.random.default_rng(seed).choice(rust, min(n, len(rust)), replace=False))
    files = [str(counts['files'][i]) for i in idx]
    ds = D.RustDataset(cfg.data_root, 'test', files, D.eval_transform(cfg.img_size))
    imgs = torch.stack([ds[i][0] for i in range(len(ds))])
    masks = torch.stack([ds[i][1] for i in range(len(ds))])

    preds = {}
    for name, run_dir in run_dirs.items():
        model = build_model(name, pretrained=False, dropout=cfg.dropout).to(device).eval()
        load_checkpoint(model, os.path.join(run_dir, 'best.pt'), map_location=device)
        with open(os.path.join(run_dir, 'metrics.json')) as f:
            thr = json.load(f)['threshold']
        with torch.no_grad():
            preds[name] = (torch.sigmoid(model(imgs.to(device))).cpu() >= thr, thr)
        del model

    mean, std = torch.tensor(D.MEAN)[:, None, None], torch.tensor(D.STD)[:, None, None]
    cols = ['Image', 'Ground truth'] + [f'{k} (t={t:.2f})' for k, (_, t) in preds.items()]
    fig, axes = plt.subplots(
        len(files), len(cols), figsize=(3.2 * len(cols), 3.2 * len(files)), squeeze=False
    )
    for r in range(len(files)):
        axes[r][0].imshow((imgs[r] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[r][1].imshow(masks[r, 0].numpy(), cmap='gray')
        for c, (p, _) in enumerate(preds.values(), start=2):
            axes[r][c].imshow(p[r, 0].numpy(), cmap='gray')
        for c, ax in enumerate(axes[r]):
            ax.axis('off')
            if r == 0:
                ax.set_title(cols[c], fontsize=10)
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


In [ ]:
%%writefile code/nwrd/benchmark.py
"""Inference efficiency: parameters, FLOPs, latency, throughput and peak memory.

Every model is measured in the same process on the same device with random weights
(weights do not affect cost).
"""

import json
import platform
import time

import torch

from .models import MODELS, build_model


def count_flops(model, size: int, device) -> int:
    from torch.utils.flop_counter import FlopCounterMode

    x = torch.randn(1, 3, size, size, device=device)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    return fc.get_total_flops()


def time_forward(model, x, device, amp: bool, warmup: int, iters: int) -> float:
    """Median wall time per forward, in milliseconds."""
    times = []
    with torch.no_grad(), torch.autocast(device.type, enabled=amp):
        for i in range(warmup + iters):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            if i >= warmup:
                times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    return times[len(times) // 2]


def benchmark_model(name: str, size: int = 512, batches=(8,), device=None, quick: bool = False):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_model(name, pretrained=False).to(device).eval()
    params = sum(p.numel() for p in model.parameters())
    res = {
        'model': name,
        'description': MODELS[name],
        'params_M': params / 1e6,
        'GFLOPs': count_flops(model, size, device) / 1e9,
        'input': f'{size}x{size}',
    }
    res['GMACs'] = res['GFLOPs'] / 2

    if device.type == 'cuda':
        warm, iters = (3, 5) if quick else (20, 100)
        x1 = torch.randn(1, 3, size, size, device=device)
        res['gpu_latency_ms_bs1_fp32'] = time_forward(model, x1, device, False, warm, iters)
        res['gpu_latency_ms_bs1_fp16'] = time_forward(model, x1, device, True, warm, iters)
        for batch in batches:
            xb = torch.randn(batch, 3, size, size, device=device)
            torch.cuda.reset_peak_memory_stats()
            ms = time_forward(model, xb, device, True, warm, iters)
            res[f'gpu_throughput_img_s_bs{batch}_fp16'] = batch * 1000 / ms
            res[f'gpu_peak_mem_MB_bs{batch}_fp16'] = torch.cuda.max_memory_allocated() / 2**20
            del xb

    cpu = torch.device('cpu')
    model = model.to(cpu)
    warm, iters = (1, 2) if quick else (3, 10)
    res['cpu_latency_ms_bs1_fp32'] = time_forward(
        model, torch.randn(1, 3, size, size), cpu, False, warm, iters
    )
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return res


def run_benchmarks(
    models, out_path: str, size: int = 512, quick: bool = False, log=print, batches=(8,)
):
    """Benchmarks every model and writes {'env': ..., 'results': [...]} to `out_path`."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    env = {
        'device': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu',
        'cpu': platform.processor() or platform.machine(),
        'cpu_threads': torch.get_num_threads(),
        'torch': torch.__version__,
    }
    rows = []
    for name in models:
        r = benchmark_model(name, size=size, batches=batches, device=device, quick=quick)
        rows.append(r)
        log(
            f'[bench] {name}: {r["params_M"]:.2f}M params, {r["GFLOPs"]:.1f} GFLOPs, '
            + ', '.join(f'{k} {v:.1f}' for k, v in r.items() if k.startswith(('gpu_', 'cpu_')))
        )
    with open(out_path, 'w') as f:
        json.dump({'env': env, 'results': rows}, f, indent=2)
    return rows


In [ ]:
%%writefile code/nwrd/compare.py
"""Aggregates the finished runs of one experiment into its comparison report.

Outputs in <out_dir>/: comparison.md, comparison.json, figures/training_curves.png and a
summary.md per model (<out_dir>/<model>/summary.md). Every
number is computed from files the pipeline saves per run (metrics.json, history.json,
test_counts.npz), so a report can be rebuilt without checkpoints or images.
"""

import glob
import json
import os

import numpy as np

from .metrics import dice, iou
from .models import MODELS

METRICS = ('IoU', 'Dice', 'Precision', 'Recall', 'Specificity', 'Accuracy')

# Published NWRD paper results (Anwar et al., Sensors 2023, doi:10.3390/s23156942), UNet + APF.
# Evaluated on 128px patches of 10 held-out full images; their IoU and F1 are not mutually
# consistent (likely per-image averaging), so treat them as approximate reference points.
PAPER = [
    {
        'table': 'Table 3',
        'test_set': 'full test set',
        'split': 'test@thr',  # our matching test set
        'scores': {'Precision': 0.506, 'Recall': 0.624, 'Dice': 0.557},
    },
    {
        'table': 'Table 4',
        'test_set': 'background-only patches removed',
        'split': 'test_rust_patches@thr',
        'scores': {'Precision': 0.593, 'Recall': 0.552, 'Dice': 0.564, 'IoU': 0.438},
    },
]
OURS = 'canet_b4'

VERDICTS = {
    'improves': 'improves on',
    'comparable': 'is comparable to',
    'worse': 'is worse than',
}


def _read_json(path: str):
    if not os.path.exists(path):
        return None
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def load_runs(out_dir: str) -> dict:
    """{model: {seed: run}} for every run with a metrics.json. A run holds its directory and
    metrics, plus 'history' (the whole history.json), 'sweep' (val threshold sweep) and
    'files' / 'counts' / 'counts05' (per-patch test confusion counts) where those files exist."""
    runs = {}
    for path in sorted(glob.glob(os.path.join(out_dir, '*', 'seed*', 'metrics.json'))):
        run_dir = os.path.dirname(path)
        model = os.path.basename(os.path.dirname(run_dir))
        seed = int(os.path.basename(run_dir)[len('seed') :])
        run = {'dir': run_dir, 'metrics': _read_json(path)}
        for key, name in (('history', 'history.json'), ('sweep', 'val_threshold_sweep.json')):
            value = _read_json(os.path.join(run_dir, name))
            if value is not None:
                run[key] = value
        counts_path = os.path.join(run_dir, 'test_counts.npz')
        if os.path.exists(counts_path):
            with np.load(counts_path) as c:
                run['files'] = c['files']
                run['counts'] = c['at_thr'].astype(np.int64)
                run['counts05'] = c['at_05'].astype(np.int64)
        runs.setdefault(model, {})[seed] = run
    return runs


def source_photo(filename) -> str:
    """NWRD patches are named <photo>_patch_<n>_x<col>_y<row>.png (tiles add _t<rc>)."""
    return str(filename).split('_patch_')[0]


def paired_bootstrap(
    ours: dict,
    base: dict,
    seeds: list,
    n_boot: int = 5000,
    rng_seed: int = 0,
    by_photo: bool = False,
) -> dict:
    """Paired bootstrap of the seed-averaged difference in pooled test IoU/Dice.

    Each resample draws test patches with replacement, or with by_photo whole source
    photographs with all their patches. Patches of one photograph are correlated (same plant,
    light and annotator), so by_photo is the honest interval; the patch-level one is narrower
    than the evidence supports. The same draws are used for both models and every seed."""
    for s in seeds:
        if not np.array_equal(ours[s]['files'], base[s]['files']):
            raise ValueError(f'seed {s}: the two models were tested on different files')
    files = ours[seeds[0]]['files']
    n = len(files)
    rng = np.random.default_rng(rng_seed)
    if by_photo:
        photos, inv = np.unique([source_photo(f) for f in files], return_inverse=True)
        draws = rng.integers(0, len(photos), size=(n_boot, len(photos)))
        w = np.stack([np.bincount(d, minlength=len(photos)) for d in draws])[:, inv]
        units = len(photos)
    else:
        w = rng.multinomial(n, np.full(n, 1 / n), size=n_boot)
        units = n
    out = {}
    for name, fn in (('IoU', iou), ('Dice', dice)):
        d = np.mean([fn(w @ ours[s]['counts']) - fn(w @ base[s]['counts']) for s in seeds], 0)
        point = np.mean(
            [fn(ours[s]['counts'].sum(0)) - fn(base[s]['counts'].sum(0)) for s in seeds]
        )
        out[name] = {
            'delta': float(point),
            'ci95': [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            'p_two_sided': float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))),
            'units': int(units),
        }
    return out


def seed_ttest(ours: dict, base: dict, seeds: list) -> dict | None:
    """Paired t-test over seeds of test IoU; it captures training-run variance."""
    if len(seeds) < 2:
        return None
    from scipy import stats

    a = [ours[s]['metrics']['test@thr']['IoU'] for s in seeds]
    b = [base[s]['metrics']['test@thr']['IoU'] for s in seeds]
    t, p = stats.ttest_rel(a, b)
    return {'t': float(t), 'p': float(p), 'n_seeds': len(seeds)}


def seed_wins(ours: dict, base: dict, seeds: list) -> int:
    """Number of seeds on which `ours` has the higher test IoU."""
    return sum(
        ours[s]['metrics']['test@thr']['IoU'] > base[s]['metrics']['test@thr']['IoU'] for s in seeds
    )


def verdict(boot_photo_iou: dict, wins: int) -> str:
    """The decision rule, fixed before the results were seen (docs/protocol.md).

    'improves': higher mean test IoU, the 95% CI of the IoU difference that resamples test
    photographs excludes 0, and CANet wins on at least 2 of 3 seeds. 'worse': that CI lies
    entirely below 0. 'comparable': otherwise. The rule assumes all three seeds; with fewer,
    the reading is interim."""
    lo, hi = boot_photo_iou['ci95']
    if boot_photo_iou['delta'] > 0 and lo > 0 and wins >= 2:
        return 'improves'
    if hi < 0:
        return 'worse'
    return 'comparable'


def significance(runs: dict) -> dict:
    """CANet-B4 against every other model, on the seeds both have."""
    tests = {}
    if OURS not in runs:
        return tests
    for base in (m for m in runs if m != OURS):
        seeds = sorted(set(runs[OURS]) & set(runs[base]))
        if not seeds:
            continue
        photo = paired_bootstrap(runs[OURS], runs[base], seeds, by_photo=True)
        wins = seed_wins(runs[OURS], runs[base], seeds)
        tests[base] = {
            'seeds': seeds,
            'bootstrap': paired_bootstrap(runs[OURS], runs[base], seeds),
            'bootstrap_photo': photo,
            'seed_ttest': seed_ttest(runs[OURS], runs[base], seeds),
            'canet_wins': wins,
            'verdict': verdict(photo['IoU'], wins),
        }
    return tests


def mean_std(vals) -> tuple[float, float]:
    """Mean and sample standard deviation (0 for a single value)."""
    vals = np.asarray(vals, float)
    return float(vals.mean()), float(vals.std(ddof=1)) if len(vals) > 1 else 0.0


def _fmt(m: float, s: float, n: int) -> str:
    return f'{m:.4f} ± {s:.4f}' if n > 1 else f'{m:.4f}'


def summarise(runs: dict) -> dict:
    summary = {}
    for model, seeds in runs.items():
        ms = [seeds[s]['metrics'] for s in sorted(seeds)]
        hs = [seeds[s]['history']['history'] for s in sorted(seeds)]
        row = {'seeds': sorted(seeds), 'n': len(ms)}
        for split in ('test@thr', 'test@0.5', 'val@thr', 'test_rust_patches@thr'):
            row[split] = {k: mean_std([m[split][k] for m in ms]) for k in METRICS}
        row['threshold'] = mean_std([m['threshold'] for m in ms])
        row['threshold_at_grid_edge'] = any(m['threshold_at_grid_edge'] for m in ms)
        row['best_epoch'] = [m['best_epoch'] for m in ms]
        row['train_min_per_epoch'] = mean_std([np.mean(h['epoch_seconds']) / 60 for h in hs])
        summary[model] = row
    return summary


def write_markdown(path: str, summary: dict, tests: dict, bench, test_info: dict, tiling=None):
    order = [m for m in (OURS,) if m in summary] + sorted(m for m in summary if m != OURS)
    L = [
        '# CANet-B4 vs baselines: comparison',
        '',
        f'Test split: {test_info["test_images"]} patches, {test_info["test_rust_images"]} with '
        f'rust, {100 * test_info["test_rust_pixel_fraction"]:.1f}% rust pixels. Threshold chosen '
        'per run on val (IoU-maximising, grid 0.05–0.95), then frozen for test. Metrics pooled '
        'over all test pixels; mean ± sample std over seeds.',
        '',
        '## Accuracy (test split, val-selected threshold)',
        '',
        '| Model | Seeds | ' + ' | '.join(METRICS) + ' | IoU @0.5 | Threshold |',
        '|---|---|' + '---|' * (len(METRICS) + 2),
    ]
    for m in order:
        r = summary[m]
        cells = [_fmt(*r['test@thr'][k], r['n']) for k in METRICS]
        thr = f'{r["threshold"][0]:.2f}' + (' ⚠ edge' if r['threshold_at_grid_edge'] else '')
        name = f'**{m}** (ours)' if m == OURS else m
        L.append(
            f'| {name} | {r["n"]} | '
            + ' | '.join(cells)
            + f' | {_fmt(*r["test@0.5"]["IoU"], r["n"])} | {thr} |'
        )
    L += [
        '',
        '## Against the NWRD paper (UNet + APF)',
        '',
        '| Result | Precision | Recall | F1 / Dice | IoU |',
        '|---|---|---|---|---|',
    ]
    for p in PAPER:
        s = p['scores']
        L.append(
            f'| Paper, {p["table"]}, {p["test_set"]} | '
            + ' | '.join(
                f'{s[k]:.3f}' if k in s else '–' for k in ('Precision', 'Recall', 'Dice', 'IoU')
            )
            + ' |'
        )
    for m in order:
        r = summary[m]
        for split, label in (
            ('test@thr', 'all test patches'),
            ('test_rust_patches@thr', 'rust-positive test patches (≈ Table 4)'),
        ):
            L.append(
                f'| {m}, re-run: {label} | '
                + ' | '.join(
                    f'{r[split][k][0]:.3f}' for k in ('Precision', 'Recall', 'Dice', 'IoU')
                )
                + ' |'
            )
    if tiling:
        L += [
            '',
            f'Every model here sees {tiling["tile"]}px tiles re-cut at {tiling["scale"]:g}× '
            "scale, the paper's own patch size and field of view (nwrd/tiles.py). The paper's "
            'test photographs, training (APF, 500 epochs, focal loss) and patch sampling still '
            'differ, so its rows are reference points, not a paired comparison. The `unet` row is '
            "the paper's architecture re-trained under this protocol, so CANet vs `unet` is the "
            'like-for-like comparison.',
        ]
    else:
        L += [
            '',
            'Paper figures are as published, on a different test unit (full-image patches, '
            "128px). The `unet` row is the paper's architecture re-trained under this protocol, so "
            'CANet vs `unet` is the like-for-like comparison.',
        ]
    if tests:

        def ci(b):
            return f'[{b["ci95"][0]:+.4f}, {b["ci95"][1]:+.4f}] (p {b["p_two_sided"]:.4f})'

        L += [
            '',
            '## Is the difference real? (CANet-B4 minus baseline, test split)',
            '',
            '| Baseline | Paired seeds | ΔIoU | 95% CI, resampling photographs | '
            '95% CI, resampling patches | ΔDice | ΔDice 95% CI, photographs | '
            'seed-level t-test p | CANet higher (seeds) | Verdict |',
            '|---|---|---|---|---|---|---|---|---|---|',
        ]
        for base, t in tests.items():
            bp, tt = t['bootstrap_photo'], t['seed_ttest']
            n = len(t['seeds'])
            reading = f'CANet-B4 {VERDICTS[t["verdict"]]} {base}'
            if n < 3:
                reading += f' (interim: {n} of 3 seeds)'
            p_seeds = f'{tt["p"]:.4f}' if tt else 'n/a (1 seed)'
            L.append(
                f'| {base} | {n} | {bp["IoU"]["delta"]:+.4f} | {ci(bp["IoU"])} | '
                f'{ci(t["bootstrap"]["IoU"])} | {bp["Dice"]["delta"]:+.4f} | {ci(bp["Dice"])} | '
                f'{p_seeds} | {t["canet_wins"]} of {n} | {reading} |'
            )
        unets = [m for m in ('unet', 'unet_tuned') if m in summary]
        if len(unets) == 2:
            best = max(unets, key=lambda m: summary[m]['val@thr']['IoU'][0])
            L += [
                '',
                "Two runs of the paper's UNet exist (shared lr and `unet_tuned`). The stronger on "
                f'**validation** IoU, `{best}`, is the paper-architecture baseline; choosing by '
                'test IoU would bias the comparison.',
            ]
        L += [
            '',
            'Verdict: the decision rule fixed before the results were seen. CANet-B4 *improves on* '
            'a baseline if its mean test IoU is higher, the photograph-resampling 95% CI of ΔIoU '
            'excludes 0, and it is higher on at least 2 of 3 seeds; it *is worse* if that CI lies '
            'below 0; otherwise the two are *comparable*.',
            '',
            'A CI that excludes 0 means the gap is larger than test-set sampling noise. Use the '
            'photograph interval: the test patches are cut from a few photographs, and patches of '
            'one photograph are not independent, so resampling patches understates the '
            'uncertainty. Neither bootstrap captures training-run variance; the seed-level t-test '
            'does, but has little power with 3 seeds.',
        ]
    if bench:
        env, rows = bench['env'], {r['model']: r for r in bench['results']}
        first = next(iter(rows.values()))
        gpu_keys = [k for k in first if k.startswith('gpu_')]
        L += [
            '',
            f'## Efficiency ({env["device"]}, {first["input"]} input, torch {env["torch"]})',
            '',
            '| Model | Params (M) | GFLOPs | '
            + ' | '.join(k[4:] for k in gpu_keys)
            + ' | CPU latency bs1 (ms) | Train min/epoch |',
            '|---|---|---|' + '---|' * (len(gpu_keys) + 2),
        ]
        for m in [x for x in dict.fromkeys(order + list(rows)) if x in rows]:
            r = rows[m]
            tr = summary.get(m, {}).get('train_min_per_epoch')
            L.append(
                f'| {m} | {r["params_M"]:.2f} | {r["GFLOPs"]:.1f} | '
                + ' | '.join(f'{r[k]:.1f}' for k in gpu_keys)
                + f' | {r["cpu_latency_ms_bs1_fp32"]:.0f} | {f"{tr[0]:.1f}" if tr else "–"} |'
            )
        if 'canet_b4_twopass' in rows:
            L += [
                '',
                '`canet_b4_twopass` is CANet as first implemented, running its encoder twice per '
                'forward pass; its outputs are bit-identical to `canet_b4`, so only the cost '
                'differs. It was measured in this run and is no longer part of the code.',
            ]
    L += ['', '## Models', ''] + [f'- `{m}` — {MODELS.get(m, m)}' for m in order]
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(L) + '\n')


def write_model_summaries(out_dir: str, runs: dict, summary: dict, tests: dict, bench) -> None:
    """Writes <out_dir>/<model>/summary.md: one model's runs, seed by seed, its mean ± std, its
    comparison with CANet-B4 and its efficiency, so each model's results stand on their own."""
    rows = {r['model']: r for r in bench['results']} if bench else {}
    keys = ('IoU', 'Dice', 'Precision', 'Recall')
    head = (
        '| Seed | Test IoU | Dice | Precision | Recall | IoU @0.5 | IoU, rust patches | Val IoU '
        '| Threshold | Best epoch | Train min/epoch |'
    )
    for model, seeds in runs.items():
        any_m = next(iter(seeds.values()))['metrics']
        L = [
            f'# {model}: {MODELS.get(model, model)}',
            '',
            f'Test split: {any_m["test_images"]} patches, {any_m["test_rust_images"]} with rust. '
            'Threshold chosen per run on validation; metrics pooled over all test pixels.',
            '',
            head,
            '|---|' + '---|' * 10,
        ]
        for s in sorted(seeds):
            m, h = seeds[s]['metrics'], seeds[s]['history']['history']
            t = m['test@thr']
            L.append(
                f'| {s} | '
                + ' | '.join(f'{t[k]:.4f}' for k in keys)
                + f' | {m["test@0.5"]["IoU"]:.4f} | {m["test_rust_patches@thr"]["IoU"]:.4f} | '
                f'{m["val@thr"]["IoU"]:.4f} | {m["threshold"]:.2f} | {m["best_epoch"]} | '
                f'{np.mean(h["epoch_seconds"]) / 60:.1f} |'
            )
        r = summary[model]
        means = [r['test@thr'][k] for k in keys] + [
            r['test@0.5']['IoU'],
            r['test_rust_patches@thr']['IoU'],
            r['val@thr']['IoU'],
        ]
        L.append(
            '| **mean ± std** | '
            + ' | '.join(_fmt(*v, r['n']) for v in means)
            + f' | {r["threshold"][0]:.2f} | – | {r["train_min_per_epoch"][0]:.1f} |'
        )
        pairs = [(base, t) for base, t in tests.items() if model in (OURS, base)]
        if pairs:
            L += ['', '## Against CANet-B4 (CANet-B4 minus baseline, pre-registered rule)', '']
            for base, t in pairs:
                b = t['bootstrap_photo']['IoU']
                L.append(
                    f'- vs `{base}`: ΔIoU {b["delta"]:+.4f}, 95% CI [{b["ci95"][0]:+.4f}, '
                    f'{b["ci95"][1]:+.4f}] resampling test photographs; CANet-B4 higher on '
                    f'{t["canet_wins"]} of {len(t["seeds"])} seeds: CANet-B4 '
                    f'**{VERDICTS[t["verdict"]]}** `{base}`.'
                )
        if model in rows:
            e = rows[model]
            gpu = ', '.join(f'{k[4:]} {v:.1f}' for k, v in e.items() if k.startswith('gpu_'))
            L += [
                '',
                f'## Efficiency ({bench["env"]["device"]}, {e["input"]} input)',
                '',
                f'{e["params_M"]:.2f}M parameters, {e["GFLOPs"]:.2f} GFLOPs; {gpu}; CPU latency '
                f'bs1 {e["cpu_latency_ms_bs1_fp32"]:.0f} ms.',
            ]
        with open(os.path.join(out_dir, model, 'summary.md'), 'w', encoding='utf-8') as f:
            f.write('\n'.join(L) + '\n')


def plot_figures(runs: dict, fig_dir: str) -> None:
    """Validation loss and IoU per epoch: mean over seeds, band from min to max."""
    import matplotlib

    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    os.makedirs(fig_dir, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for model, seeds in sorted(runs.items()):
        for key, ax in (('val_loss', axes[0]), ('val_iou', axes[1])):
            curves = [seeds[s]['history']['history'][key] for s in sorted(seeds)]
            n = min(map(len, curves))
            arr = np.array([c[:n] for c in curves])
            x = np.arange(1, n + 1)
            (line,) = ax.plot(x, arr.mean(0), label=model, lw=2 if model == OURS else 1.4)
            if len(curves) > 1:
                ax.fill_between(x, arr.min(0), arr.max(0), color=line.get_color(), alpha=0.15)
    axes[0].set_title('Validation loss')
    axes[1].set_title('Validation IoU (pooled, threshold 0.5)')
    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(fig_dir, 'training_curves.png'), dpi=150)
    plt.close(fig)


def compare(out_dir: str, log=print) -> dict | None:
    """Writes comparison.md, comparison.json and the training-curve figure for `out_dir`."""
    runs = load_runs(out_dir)
    if not runs:
        log('[compare] no finished runs yet')
        return None
    incomplete = [
        r['dir']
        for seeds in runs.values()
        for r in seeds.values()
        if not {'history', 'counts'} <= set(r)
    ]
    if incomplete:
        raise FileNotFoundError(f'history.json or test_counts.npz missing in {incomplete}')
    summary = summarise(runs)
    tests = significance(runs)
    bench = _read_json(os.path.join(out_dir, 'benchmark.json'))
    tiling = _read_json(os.path.join(out_dir, 'tiling.json'))
    any_run = next(iter(next(iter(runs.values())).values()))['metrics']
    plot_figures(runs, os.path.join(out_dir, 'figures'))
    md_path = os.path.join(out_dir, 'comparison.md')
    write_markdown(md_path, summary, tests, bench, any_run, tiling)
    write_model_summaries(out_dir, runs, summary, tests, bench)
    with open(os.path.join(out_dir, 'comparison.json'), 'w') as f:
        json.dump({'summary': summary, 'significance': tests, 'benchmark': bench}, f, indent=2)
    with open(md_path, encoding='utf-8') as f:
        log(f.read())
    return summary


In [ ]:
%%writefile code/nwrd/export.py
"""Packages the model the app serves: the CANet-B4 seed with the best *validation* IoU.

Writes <out_dir>/export/: best_model.pth, results.json (test metrics and training history),
confusion_matrix.png, and copies of the experiment's figures. Selection never looks at test
metrics. Copy the folder's files to app/assets/ to update the app.
"""

import json
import os
import shutil

import numpy as np

from .compare import OURS, load_runs


def plot_confusion(counts: np.ndarray, thr: float, path: str) -> None:
    import matplotlib

    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    tp, fp, fn, tn = counts.sum(0)
    cm = np.array([[tn, fp], [fn, tp]])
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(
            j, i, f'{v:,}', ha='center', va='center', color='white' if v > cm.max() / 2 else 'black'
        )
    ax.set_xticks([0, 1], ['Pred background', 'Pred rust'])
    ax.set_yticks([0, 1], ['True background', 'True rust'])
    ax.set_title(f'Test confusion matrix (threshold {thr:.2f})')
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def export_for_app(out_dir: str, log=print) -> int | None:
    """Exports the best-on-validation CANet-B4 seed; returns that seed, or None without runs."""
    runs = load_runs(out_dir).get(OURS)
    if not runs:
        log(f'[export] no finished {OURS} runs')
        return None
    seed = max(runs, key=lambda s: runs[s]['metrics']['val@thr']['IoU'])
    run = runs[seed]
    m, h = run['metrics'], run['history']['history']
    exp = os.path.join(out_dir, 'export')
    os.makedirs(exp, exist_ok=True)
    t = m['test@thr']
    results = {
        'model': OURS,
        'seed': seed,
        'split': 'test',
        'best_epoch': m['best_epoch'],
        'best_threshold': m['threshold'],
        'final_metrics': {
            'IoU': round(t['IoU'], 4),
            'F1 / Dice': round(t['Dice'], 4),
            'Precision': round(t['Precision'], 4),
            'Recall': round(t['Recall'], 4),
            'Specificity': round(t['Specificity'], 4),
            'Accuracy': round(t['Accuracy'], 4),
        },
        'history': {k: h[k] for k in ('train_loss', 'val_loss', 'val_iou', 'val_dice')},
    }
    with open(os.path.join(exp, 'results.json'), 'w') as f:
        json.dump(results, f, indent=2)
    shutil.copy(os.path.join(run['dir'], 'best.pt'), os.path.join(exp, 'best_model.pth'))
    plot_confusion(run['counts'], m['threshold'], os.path.join(exp, 'confusion_matrix.png'))
    fig_dir = os.path.join(out_dir, 'figures')
    for name in ('training_curves.png', 'qualitative_results.png'):
        if os.path.exists(os.path.join(fig_dir, name)):
            shutil.copy(os.path.join(fig_dir, name), os.path.join(exp, name))
    log(f'[export] {OURS} seed {seed} (best val IoU) -> {exp}')
    return seed


In [ ]:
%%writefile code/nwrd/run.py
"""Runs one experiment: [tile] -> train -> evaluate -> benchmark -> figures -> compare -> export.

    python -m nwrd.run --preset main --data-root /path/to/wheat_rust_patches --out runs
    python -m nwrd.run --preset 128px --data-root /path/to/wheat_rust_patches --out runs_128px
    python -m nwrd.run --stage compare --out results/512px/final     # re-aggregate only

Finished stages are skipped, and an interrupted training run resumes from last.pt, so the same
command can be repeated across Kaggle sessions until everything is done. Exit code 3 means the
time budget stopped training: run the same command again to continue.
"""

import argparse
import json
import os
import platform
import shutil
import sys
import tempfile
import time
import traceback
import typing
from dataclasses import fields

from . import __version__
from .config import PRESETS, Config

STAGES = ('train', 'eval', 'bench', 'figures', 'compare', 'export')
# Stages that create run records. Only these write config.json and log.txt, so re-aggregating
# a finished results folder (compare, export) never overwrites what the original run recorded.
RECORDING_STAGES = {'train', 'eval', 'bench', 'figures'}


def _parse_bool(s: str) -> bool:
    v = s.strip().lower()
    if v in ('1', 'true', 'yes'):
        return True
    if v in ('0', 'false', 'no'):
        return False
    raise argparse.ArgumentTypeError(f'expected true or false, got {s!r}')


def parse_args(argv=None) -> argparse.Namespace:
    """Every Config field is a flag; the preset's values are the defaults."""
    pre = argparse.ArgumentParser(add_help=False)
    pre.add_argument('--preset', choices=sorted(PRESETS), default='main')
    preset = pre.parse_known_args(argv)[0].preset
    defaults = {**Config().to_dict(), **PRESETS[preset]}
    hints = typing.get_type_hints(Config)

    p = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter
    )
    p.add_argument(
        '--preset',
        choices=sorted(PRESETS),
        default='main',
        help='the experiment whose settings are the defaults (nwrd/config.py)',
    )
    p.add_argument('--stage', default='all', choices=('all',) + STAGES)
    p.add_argument('--out', dest='out_dir', default=defaults['out_dir'])
    for f in fields(Config):
        if f.name == 'out_dir':
            continue
        flag, hint, default = '--' + f.name.replace('_', '-'), hints[f.name], defaults[f.name]
        if hint is bool:
            p.add_argument(
                flag, dest=f.name, type=_parse_bool, default=default, metavar='{true,false}'
            )
        elif typing.get_origin(hint) is list:
            p.add_argument(
                flag, dest=f.name, nargs='+', type=typing.get_args(hint)[0], default=default
            )
        else:
            p.add_argument(flag, dest=f.name, type=hint, default=default)
    return p.parse_args(argv)


def _environment() -> str:
    """Library versions and device, recorded at the top of every session's log."""
    from importlib.metadata import PackageNotFoundError, version

    import cv2
    import numpy
    import torch

    def v(package):
        try:
            return version(package)
        except PackageNotFoundError:
            return 'not installed'

    device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    return (
        f'python {platform.python_version()}, torch {torch.__version__}, '
        f'numpy {numpy.__version__}, opencv {cv2.__version__}, '
        f'albumentations {v("albumentations")}, '
        f'segmentation-models-pytorch {v("segmentation_models_pytorch")}, '
        f'efficientnet-pytorch {v("efficientnet_pytorch")}, device {device}'
    )


def _benchmarked(out_dir: str, models: list[str]) -> bool:
    """Whether out_dir/benchmark.json already has a row for every model."""
    path = os.path.join(out_dir, 'benchmark.json')
    if not os.path.exists(path):
        return False
    with open(path) as f:
        done = {r['model'] for r in json.load(f)['results']}
    return set(models) <= done


def main(argv=None) -> int:
    t_session = time.time()  # the time budget counts from here, across all runs
    a = parse_args(argv)
    cfg = Config(**{f.name: getattr(a, f.name) for f in fields(Config)})
    cfg.validate()
    stages = STAGES if a.stage == 'all' else (a.stage,)
    os.makedirs(cfg.out_dir, exist_ok=True)
    recording = bool(RECORDING_STAGES & set(stages))
    log_file = (
        open(os.path.join(cfg.out_dir, 'log.txt'), 'a', encoding='utf-8') if recording else None
    )

    def log(msg: str) -> None:
        line = f'{time.strftime("%H:%M:%S")} {msg}'
        print(line, flush=True)
        if log_file:
            log_file.write(line + '\n')
            log_file.flush()

    try:
        return _run(cfg, a.preset, stages, log, t_session, recording)
    except Exception:
        # On Kaggle the notebook's own output cannot be downloaded, but runs/log.txt can.
        if log_file:
            log_file.write(f'{time.strftime("%H:%M:%S")} CRASHED:\n{traceback.format_exc()}')
        raise
    finally:
        if log_file:
            log_file.close()


def _run(cfg: Config, preset: str, stages: tuple, log, t_session: float, recording: bool) -> int:
    log(f'nwrd {__version__} | preset {preset} | stages {",".join(stages)} | {_environment()}')
    if {'train', 'eval', 'figures'} & set(stages):
        from .data import find_data_root

        cfg.data_root = find_data_root(cfg.data_root or None)
        log(f'data: {cfg.data_root}')
        if cfg.tile_size:
            # Tiles are rebuilt each session outside out_dir, so they never enter the saved output.
            from .tiles import build_tiles

            tile_dir = cfg.tile_dir or os.path.join(
                tempfile.gettempdir(), f'nwrd_tiles_{cfg.tile_size}px_x{cfg.tile_scale:g}'
            )
            cfg.data_root = build_tiles(
                cfg.data_root, tile_dir, cfg.tile_scale, cfg.tile_size, log=log
            )
            shutil.copy(
                os.path.join(tile_dir, 'tiling.json'), os.path.join(cfg.out_dir, 'tiling.json')
            )
            log(f'data: {cfg.tile_size}px tiles at {cfg.tile_scale:g}x in {cfg.data_root}')
    if recording:
        with open(os.path.join(cfg.out_dir, 'config.json'), 'w') as f:
            json.dump({'preset': preset, **cfg.to_dict()}, f, indent=2)

    # Seed-major order: after each seed finishes, every model has a paired run to compare.
    # unet_tuned is a control (is the shared lr fair to the scratch UNet?), so its runs go last:
    # if sessions or quota run out, the main comparison is the part that is complete.
    runs = [(m, s, os.path.join(cfg.out_dir, m, f'seed{s}')) for s in cfg.seeds for m in cfg.models]
    runs.sort(key=lambda r: r[0] == 'unet_tuned')
    complete, session_start = True, None
    for model, seed, run_dir in runs:
        if 'train' in stages and not os.path.exists(os.path.join(run_dir, 'TRAINED')):
            from .train import train_run

            if not train_run(cfg, model, seed, run_dir, log=log, session_start=session_start):
                complete = False
                break  # out of time budget: stop here, resume next session
            session_start = t_session
        if (
            'eval' in stages
            and os.path.exists(os.path.join(run_dir, 'TRAINED'))
            and not os.path.exists(os.path.join(run_dir, 'metrics.json'))
        ):
            from .evaluate import evaluate_run

            evaluate_run(cfg, model, run_dir, log=log)

    bench_models = cfg.bench_models or cfg.models
    if 'bench' in stages and not _benchmarked(cfg.out_dir, bench_models):
        from .benchmark import run_benchmarks

        # Every row of the table is measured in one session on one device, so a model added
        # later re-measures them all rather than mixing sessions.
        run_benchmarks(
            bench_models,
            os.path.join(cfg.out_dir, 'benchmark.json'),
            size=cfg.img_size,
            quick=cfg.bench_quick,
            log=log,
            batches=cfg.bench_batches,
        )

    if 'figures' in stages:
        from .evaluate import plot_qualitative

        first = {m: os.path.join(cfg.out_dir, m, f'seed{cfg.seeds[0]}') for m in cfg.models}
        first = {m: d for m, d in first.items() if os.path.exists(os.path.join(d, 'metrics.json'))}
        if first:
            os.makedirs(os.path.join(cfg.out_dir, 'figures'), exist_ok=True)
            plot_qualitative(
                cfg, first, os.path.join(cfg.out_dir, 'figures', 'qualitative_results.png')
            )

    if 'compare' in stages:
        from .compare import compare

        compare(cfg.out_dir, log=log)

    if 'export' in stages:
        if cfg.tile_size:
            # The app serves full-resolution photographs to a model trained on 512px patches; a
            # model trained on tiles of half-size photographs must never replace it.
            log('[export] skipped: models trained on tiles are not for the app')
        else:
            from .export import export_for_app

            export_for_app(cfg.out_dir, log=log)

    if not complete:
        log('INCOMPLETE: re-run the same command in a new session to continue.')
    return 0 if complete else 3


if __name__ == '__main__':
    sys.exit(main())


### Data and resume

In [ ]:
from nwrd.data import find_data_root

DATA_ROOT = find_data_root(None)   # the attached dataset, under /kaggle/input
print('dataset:', DATA_ROOT, '|',
      {s: len(os.listdir(os.path.join(DATA_ROOT, s, 'images'))) for s in ('train', 'val', 'test')})
# The dataset's own notes say how it was made (padding, split by photograph).
for f in ('README.md', 'processing_stats.json'):
    p = os.path.join(DATA_ROOT, f)
    if os.path.exists(p):
        print('--- %s ---' % f)
        print(open(p, encoding='utf-8', errors='replace').read()[:3000])

In [ ]:
# Continue an earlier session. Kaggle offers the output of a finished version as an input
# (not of one killed at the session limit); its runs/ is copied into place here.
prev = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/runs/config.json', recursive=True))
if prev and not os.path.exists(os.path.join(OUT, 'config.json')):
    shutil.copytree(prev[0], OUT, dirs_exist_ok=True)
    print('restored previous runs from', prev[0])
    with open(os.path.join(OUT, 'log.txt'), 'a', encoding='utf-8') as f:
        f.write('--- new session: restored previous runs from %s\n' % prev[0])
states = {}
for d in sorted(glob.glob(os.path.join(OUT, '*', 'seed*'))):
    states[d] = ('done' if os.path.exists(os.path.join(d, 'metrics.json')) else
                 'trained' if os.path.exists(os.path.join(d, 'TRAINED')) else
                 'partial' if os.path.exists(os.path.join(d, 'last.pt')) else 'empty')
    print(' ', os.path.relpath(d, OUT), states[d])
print('%d run directories found' % len(states))
if EXPECT_RESTORE and not any(s != 'empty' for s in states.values()):
    print('/kaggle/input:', sorted(glob.glob('/kaggle/input/*/*/*'))[:30])
    raise RuntimeError('Resume was requested but no earlier runs/ was found under /kaggle/input. '
                       'Stopping now instead of retraining every run from scratch.')

### Run the experiment

In [ ]:
cmd = [sys.executable, '-m', 'nwrd.run', '--preset', PRESET, '--data-root', DATA_ROOT,
       '--out', OUT, '--time-budget-hours', str(KAGGLE_TIME_BUDGET_HOURS),
       '--num-workers', str(NUM_WORKERS)]
print(' '.join(cmd))
proc = subprocess.Popen(cmd, cwd='code', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
code = proc.wait()
print('exit code', code, '(3 = stopped at the time budget; commit again to resume)')
# Fail the version, so Kaggle reports ERROR instead of a COMPLETE run that trained nothing.
if code not in (0, 3):
    raise RuntimeError('the pipeline failed (exit code %d); the traceback is in runs/log.txt' % code)

In [ ]:
from IPython.display import Image, Markdown, display
if os.path.exists(f'{OUT}/comparison.md'):
    display(Markdown(open(f'{OUT}/comparison.md', encoding='utf-8').read()))
    for f in ('training_curves.png', 'qualitative_results.png'):
        if os.path.exists(f'{OUT}/figures/{f}'):
            display(Image(f'{OUT}/figures/{f}'))

## Taking the results back to the repository

`python scripts/run_on_kaggle.py --experiment 128px --status` downloads this version's output to
`runs_kaggle/128px/<account>_v<N>/` (weights stay on Kaggle). Copy its `runs/` to
`results/128px/`, without checkpoints or `TRAINED` markers.

`runs/tiling.json` records how the tiles were made and their rust-pixel share before and after
downsampling. No model is exported: these models expect 128px tiles of half-size photographs,
while the app serves full-resolution photographs to the 512px model.